# Exercise 1 — Simple: Multinomial Naive Bayes Spam Classifier


## Objective
Build a **Multinomial Naive Bayes** spam classifier **completely from scratch** using only Python's built-in libraries

## Key Formula
$$\hat{y} = \arg\max_C \left[ \log P(C) + \sum_i \log P(w_i \mid C) \right]$$

With **Laplace smoothing** (α = 1):
$$P(w \mid C) = \frac{\text{count}(w, C) + \alpha}{\text{totalWords}(C) + \alpha \cdot |V|}$$

## What You'll Build
1. Tokenize text into word lists  
2. Count word occurrences per class  
3. Apply Laplace smoothing  
4. Compute log-probabilities  
5. Classify a new sentence  
6. Print prediction with scores


## Cell 1 — Training Data
Four short emails labelled SPAM or HAM. We'll learn word distributions from these.

In [4]:
import math
import re
from collections import defaultdict

# ── Training corpus ────────────────────────────────────────────────────────
# Each tuple: (email_text, label)
train = [
    ("free money win prize",        "spam"),
    ("win free lottery now",         "spam"),
    ("meeting at office tomorrow",   "ham"),
    ("hi how are you today",         "ham"),
]

# Simple tokenizer: lowercase + split on non-word characters
tokenize = lambda s: re.findall(r'\w+', s.lower())

print("Training corpus:")
for text, label in train:
    print(f"  [{label.upper():4s}]  '{text}'")


Training corpus:
  [SPAM]  'free money win prize'
  [SPAM]  'win free lottery now'
  [HAM ]  'meeting at office tomorrow'
  [HAM ]  'hi how are you today'


## Cell 2 — Count Words per Class
Build:
- `class_counts` → number of documents per class
- `word_counts[class][word]` → how often each word appears in that class
- `vocab` → set of ALL unique words across all classes


In [5]:
# ── Step 1: Count documents and word occurrences per class ─────────────────
class_counts = defaultdict(int)                         # {label: num_docs}
word_counts  = defaultdict(lambda: defaultdict(int))    # {label: {word: count}}

for text, label in train:
    class_counts[label] += 1
    for word in tokenize(text):
        word_counts[label][word] += 1

# ── Build vocabulary (union of all words) ───────────────────────────────────
vocab = set(word for text, _ in train for word in tokenize(text))
V     = len(vocab)                   # vocabulary size |V|
N     = sum(class_counts.values())   # total number of documents

print(f"Vocabulary ({V} words): {sorted(vocab)}")
print(f"\nClass document counts: {dict(class_counts)}")
print(f"\nWord counts per class:")
for cls in class_counts:
    print(f"  [{cls.upper()}]: {dict(word_counts[cls])}")


Vocabulary (15 words): ['are', 'at', 'free', 'hi', 'how', 'lottery', 'meeting', 'money', 'now', 'office', 'prize', 'today', 'tomorrow', 'win', 'you']

Class document counts: {'spam': 2, 'ham': 2}

Word counts per class:
  [SPAM]: {'free': 2, 'money': 1, 'win': 2, 'prize': 1, 'lottery': 1, 'now': 1}
  [HAM]: {'meeting': 1, 'at': 1, 'office': 1, 'tomorrow': 1, 'hi': 1, 'how': 1, 'are': 1, 'you': 1, 'today': 1}


## Cell 3 — Log-Probability with Laplace Smoothing
We compute **log P(word | class)** using:
$$\log P(w \mid C) = \log\frac{\text{count}(w, C) + \alpha}{\text{totalWords}(C) + \alpha \cdot |V|}$$

Using **log** avoids floating-point underflow when multiplying many small probabilities.


In [6]:
# ── Step 2: Log-probability function with Laplace smoothing (α = 1) ────────
def log_prob_word_given_class(word, cls, alpha=1):
    """
    Returns log P(word | cls) with Laplace smoothing.

    Args:
        word  : the word token
        cls   : class label (e.g. 'spam' or 'ham')
        alpha : smoothing parameter (default 1 = Laplace)
    """
    total_words_in_cls = sum(word_counts[cls].values())
    count_word_cls     = word_counts[cls][word]   # 0 if unseen (defaultdict)

    return math.log((count_word_cls + alpha) /
                    (total_words_in_cls + alpha * V))

# ── Demonstration ────────────────────────────────────────────────────────────
print("Log-probability examples:")
for word in ["free", "money", "meeting", "lottery", "UNSEEN_WORD"]:
    for cls in ["spam", "ham"]:
        lp = log_prob_word_given_class(word, cls)
        print(f"  log P('{word}' | {cls}) = {lp:.4f}  →  P = {math.exp(lp):.4f}")
    print()


Log-probability examples:
  log P('free' | spam) = -2.0369  →  P = 0.1304
  log P('free' | ham) = -3.1781  →  P = 0.0417

  log P('money' | spam) = -2.4423  →  P = 0.0870
  log P('money' | ham) = -3.1781  →  P = 0.0417

  log P('meeting' | spam) = -3.1355  →  P = 0.0435
  log P('meeting' | ham) = -2.4849  →  P = 0.0833

  log P('lottery' | spam) = -2.4423  →  P = 0.0870
  log P('lottery' | ham) = -3.1781  →  P = 0.0417

  log P('UNSEEN_WORD' | spam) = -3.1355  →  P = 0.0435
  log P('UNSEEN_WORD' | ham) = -3.1781  →  P = 0.0417



## Cell 4 — Classify a New Email
For each class we compute:
$$\text{score}(C) = \log P(C) + \sum_{w \in \text{query}} \log P(w \mid C)$$

The class with the **highest score** wins.


In [7]:
# ── Step 3: Classify a new query sentence ────────────────────────────────────
query = "free lottery win now"
query_words = tokenize(query)

print(f"Query: '{query}'")
print(f"Tokens: {query_words}\n")

scores = {}
for cls in class_counts:
    log_prior      = math.log(class_counts[cls] / N)           # log P(C)
    log_likelihood = sum(log_prob_word_given_class(w, cls)      # Σ log P(wᵢ|C)
                         for w in query_words)
    scores[cls] = log_prior + log_likelihood

    print(f"[{cls.upper()}]")
    print(f"  log P({cls})              = log({class_counts[cls]}/{N}) = {log_prior:.4f}")
    for w in query_words:
        lp = log_prob_word_given_class(w, cls)
        print(f"  log P('{w}' | {cls}) = {lp:.4f}")
    print(f"  Total score              = {scores[cls]:.4f}\n")

# ── Decision ─────────────────────────────────────────────────────────────────
prediction = max(scores, key=scores.get)
print("=" * 50)
print(f"Scores  →  spam: {scores['spam']:.4f}  |  ham: {scores['ham']:.4f}")
print(f"Prediction: {prediction.upper()}")


Query: 'free lottery win now'
Tokens: ['free', 'lottery', 'win', 'now']

[SPAM]
  log P(spam)              = log(2/4) = -0.6931
  log P('free' | spam) = -2.0369
  log P('lottery' | spam) = -2.4423
  log P('win' | spam) = -2.0369
  log P('now' | spam) = -2.4423
  Total score              = -9.6516

[HAM]
  log P(ham)              = log(2/4) = -0.6931
  log P('free' | ham) = -3.1781
  log P('lottery' | ham) = -3.1781
  log P('win' | ham) = -3.1781
  log P('now' | ham) = -3.1781
  Total score              = -13.4054

Scores  →  spam: -9.6516  |  ham: -13.4054
Prediction: SPAM


## Cell 5 — Visualise Scores (Optional)

In [8]:
# Simple bar chart using only built-ins — no matplotlib needed
print("\nScore comparison (higher = more likely):\n")
for cls, score in sorted(scores.items(), key=lambda x: x[1], reverse=True):
    bar_len = int((score + 10) * 3)   # scale for display
    bar = "█" * max(bar_len, 1)
    marker = " ← PREDICTED" if cls == prediction else ""
    print(f"  {cls.upper():6s} | {bar} {score:.4f}{marker}")

print()
print("─" * 50)
print("How to interpret:")
print("  Scores are log-probabilities (negative numbers).")
print("  LESS negative  = higher probability = more likely class.")



Score comparison (higher = more likely):

  SPAM   | █ -9.6516 ← PREDICTED
  HAM    | █ -13.4054

──────────────────────────────────────────────────
How to interpret:
  Scores are log-probabilities (negative numbers).
  LESS negative  = higher probability = more likely class.


## Cell 6 — Try Your Own Sentences

In [9]:
def classify(text, alpha=1):
    """Classify any text using our trained Naive Bayes model."""
    words  = tokenize(text)
    scores = {}
    for cls in class_counts:
        scores[cls] = math.log(class_counts[cls] / N)
        scores[cls] += sum(log_prob_word_given_class(w, cls, alpha) for w in words)
    pred = max(scores, key=scores.get)
    return pred, scores

# ── Test additional sentences ─────────────────────────────────────────────────
test_sentences = [
    "free money now",
    "office meeting tomorrow morning",
    "win big prize today",
    "how are you doing",
    "lottery winner free prize money",
]

print(f"{'Sentence':<40} {'Prediction':>12}   Spam Score   Ham Score")
print("-" * 75)
for sent in test_sentences:
    pred, sc = classify(sent)
    print(f"{sent:<40} {pred.upper():>12}   {sc['spam']:>10.4f}   {sc['ham']:>9.4f}")


Sentence                                   Prediction   Spam Score   Ham Score
---------------------------------------------------------------------------
free money now                                   SPAM      -7.6147    -10.2273
office meeting tomorrow morning                   HAM     -13.2351    -11.3259
win big prize today                              SPAM     -11.4434    -12.7122
how are you doing                                 HAM     -13.2351    -11.3259
lottery winner free prize money                  SPAM     -13.1926    -16.5834


## Summary

| Step | What we did |
|------|------------|
| 1 | Tokenised text into word lists |
| 2 | Counted word frequencies per class |
| 3 | Built vocabulary & computed class priors |
| 4 | Applied Laplace smoothing (α=1) to avoid zero probabilities |
| 5 | Scored each class with log-prior + Σ log-likelihoods |
| 6 | Predicted the class with the highest score |
